In [ ]:
!pip install -U sagemaker

In [ ]:
import sagemaker

In [ ]:
from sagemaker.huggingface import HuggingFace

In [ ]:
role = sagemaker.get_execution_role()

In [ ]:

hyperparameters = {
    "model_id": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    "epochs": 2,
    "per_device_train_batch_size": 2,
    "lr": 2e-5
}

In [ ]:
estimator = HuggingFace(
    entry_point="train.py",
    source_dir="./scripts",
    role=role,
    transformers_version="4.36",
    pytorch_version="2.1",
    py_version="py310",
    instance_type="ml.g5.xlarge",
    instance_count=1,
    output_path="s3://llm-model-artifacts-suman/models/",
    hyperparameters=hyperparameters
)

In [ ]:
#Run only for the training
estimator.fit({
    "train": "s3://llm-finetune-dataset-suman/datasets/"
})

In [ ]:
# estimator.latest_training_job.model_data
# estimator.model_data

In [ ]:
# Use this code to check all the accessible services inside your AWS Sagemaker

# from sagemaker import image_uris

# image_uris.retrieve(
#     framework="huggingface",
#     region="ap-south-1",   # change your region
#     version="4.37.0",
#     image_scope="inference"
# )

In [ ]:
# instance_type="ml.g5.xlarge",

In [ ]:
# model = HuggingFaceModel(
#     model_data="s3://bucket/model.tar.gz",
#     role=role,
#     entry_point="inference.py",
#     source_dir="inference",
#     transformers_version="4.36",
#     pytorch_version="2.1",
#     py_version="py310"
# )

In [ ]:
## the code for the deployment
import sagemaker
from sagemaker.huggingface import HuggingFaceModel

role = sagemaker.get_execution_role()

model = HuggingFaceModel(
    model_data="s3://llm-model-artifacts-suman/models/huggingface-pytorch-training-2026-11-15-11-42-14-536/output/model.tar.gz",
    role=role,
    transformers_version="4.37.0",
    pytorch_version="2.1.0",
    py_version="py310",
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name="live-finetune-endpoint"
)


In [ ]:
## THIS IS JUST TO VALIDATE WHETHER MODEL WORKING OR NOT IN THE NOTEBOOK ITSELF
predictor.predict({"inputs": "Explain AWS S3"})

In [ ]:
## after the deployment URL will look like this
https://runtime.sagemaker.<region>.amazonaws.com/endpoints/live-finetune-endpoint/invocations

In [ ]:
import boto3, json

runtime = boto3.client("sagemaker-runtime", region_name="ap-south-1")

resp = runtime.invoke_endpoint(
    EndpointName="live-finetune-my-endpoint",
    ContentType="application/json",
    Body=json.dumps({"inputs": "hello"})
)

print(resp["Body"].read().decode())

In [ ]:
print(predictor.predict({"inputs": "what Clinical trials have shown that adding Ezetimibe to statin therapy?"}))